In [2]:
from sentence_transformers import SentenceTransformer

print("sentence-transformers imported successfully")

sentence-transformers imported successfully


In [3]:
import torch
from sentence_transformers import SentenceTransformer

# Choose GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# Load embedding model
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

print("Model loaded successfully!")
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Embedding dimension: 384


C:\Users\User\AppData\Local\Temp\ipykernel_12056\3136126966.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [4]:
test_question = "What is Computer Vision and what problems does it solve?"

embedding = model.encode(
    test_question,
    normalize_embeddings=True
)

print("Embedding generated!")
print("Shape:", embedding.shape)
print("First 10 values:")
print(embedding[:10])

Embedding generated!
Shape: (384,)
First 10 values:
[ 0.00414769 -0.0098136   0.03878997 -0.02952867  0.01354417  0.00838838
  0.03859438 -0.0208224  -0.0557364   0.03918079]


In [5]:
questions = [
    "What is Computer Vision and what problems does it solve?",
    "What are the main applications of computer vision?",
    "How does Kubernetes manage containers?"
]

embeddings = model.encode(
    questions,
    normalize_embeddings=True
)

similarity = embeddings @ embeddings.T

print("Similarity matrix:")
print(similarity)

Similarity matrix:
[[0.99999976 0.83385247 0.48251742]
 [0.83385247 1.0000001  0.45888057]
 [0.48251742 0.45888057 1.        ]]


In [7]:
import pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

QUESTIONS_FILE = PROJECT_ROOT / "data" / "parsed_questions.pkl"

print("Loading:", QUESTIONS_FILE)

with open(QUESTIONS_FILE, "rb") as f:
    questions = pickle.load(f)

print("Loaded successfully!")
print("Number of questions:", len(questions))

Loading: C:\Users\User\RAG SYS\Question_Generation\data\parsed_questions.pkl
Loaded successfully!
Number of questions: 2150


In [13]:
def build_embedding_text(q):
    """
    Create the text representation that will be embedded.
    """

    experience = q.get("experience", [])
    skills = q.get("skills", [])
    concepts = q.get("expected_concepts", [])

    # Convert lists to text
    if isinstance(experience, list):
        experience = ", ".join(str(x) for x in experience)

    if isinstance(skills, list):
        skills = ", ".join(str(x) for x in skills)

    if isinstance(concepts, list):
        concepts = ", ".join(str(x) for x in concepts)

    return f"""
Title: {q.get('title', '')}

Question: {q.get('question', '')}

Type: {q.get('type', '')}

Track: {q.get('track', '')}

Category: {q.get('category', '')}

Topic: {q.get('topic', '')}

Difficulty: {q.get('difficulty', '')}

Experience: {experience}

Skills: {skills}

Expected Concepts: {concepts}
""".strip()


# Build embedding text for all 2,150 questions
embedding_texts = [
    build_embedding_text(q)
    for q in questions
]

print("Embedding texts created:", len(embedding_texts))

print("\n" + "=" * 70)
print("SAMPLE EMBEDDING TEXT")
print("=" * 70)

print(embedding_texts[0])

Embedding texts created: 2150

SAMPLE EMBEDDING TEXT
Title: What is Computer Vision?

Question: What is Computer Vision and what problems does it solve?

Type: technical

Track: ai_ml

Category: computer_vision

Topic: computer_vision_basics

Difficulty: easy

Experience: intern, junior

Skills: computer_vision

Expected Concepts: Image understanding, Video understanding, Computer vision systems, Image analysis, Object recognition


In [14]:
# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

print("=" * 70)
print("GENERATING EMBEDDINGS")
print("=" * 70)

embeddings = model.encode(
    embedding_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("\n" + "=" * 70)
print("EMBEDDING COMPLETE")
print("=" * 70)

print("\nNumber of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

GENERATING EMBEDDINGS


Batches:   0%|          | 0/68 [00:00<?, ?it/s]


EMBEDDING COMPLETE

Number of embeddings: 2150
Embedding shape: (2150, 384)
Data type: float32


In [15]:
# ============================================================
# COMBINE QUESTIONS + EMBEDDINGS
# ============================================================

embedded_questions = []

for i, q in enumerate(questions):

    embedded_questions.append({
        "id": q["id"],
        "embedding": embeddings[i],
        "metadata": q
    })

print("=" * 70)
print("EMBEDDED RECORDS CREATED")
print("=" * 70)

print("\nRecords:", len(embedded_questions))

print("\nFirst record:")
print("ID:", embedded_questions[0]["id"])
print("Embedding shape:", embedded_questions[0]["embedding"].shape)
print("Question:", embedded_questions[0]["metadata"]["question"])

EMBEDDED RECORDS CREATED

Records: 2150

First record:
ID: ai_ml_computer_vision_q001
Embedding shape: (384,)
Question: What is Computer Vision and what problems does it solve?


In [16]:
# ============================================================
# SAVE EMBEDDINGS
# ============================================================

import pickle
from pathlib import Path

# Question_Generation/
PROJECT_ROOT = Path.cwd().parent

# Create embeddings output directory
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Output file
OUTPUT_FILE = EMBEDDINGS_DIR / "question_embeddings.pkl"

# Save
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(embedded_questions, f)

print("=" * 70)
print("EMBEDDINGS SAVED")
print("=" * 70)

print("\nFile:")
print(OUTPUT_FILE)

print("\nRecords saved:", len(embedded_questions))

print("\n✓ Save complete")

EMBEDDINGS SAVED

File:
C:\Users\User\RAG SYS\Question_Generation\data\embeddings\question_embeddings.pkl

Records saved: 2150

✓ Save complete
